# Traitement NLP des articles (JDD)

Ce notebook décompose, **une étape à la fois**, le traitement d'un corpus d'articles de presse pour préparer l'analyse de l'évolution thématique dans le temps.

Une fois la méthode validée, elle sera généralisée aux autres journaux.

## 1. Charger le CSV et l'inspecter

On commence par lire le fichier et regarder ce qu'il y a dedans : combien d'articles,
quelles colonnes, sur quelle période. **On ne traite jamais des données qu'on n'a pas regardées.**

In [ ]:
import pandas as pd

CSV = "../data/jdd.csv"
df = pd.read_csv(CSV)

# Affichage des dimensions 
print(df.shape)

# Affichage des principales infos
print(df.info())

On convertit ensuite les dates (actuellement de type str), et on supprime les années non significatives : 

In [ ]:
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")

# On affiche le nombre d'article par année
print(df["date"].dt.year.value_counts().sort_index())

# 1990 et 2020 ne comptent qu'un seul article chacune, 2026 incomplète : on les retire (non significatif).
to_drop = df["date"].dt.year.isin([1990,2020,2026])
df = df.loc[~to_drop]

## 2. Convertir en Parquet

In [ ]:
PARQUET = "../data/jdd.parquet"

df.to_parquet(PARQUET, index=False)
df = pd.read_parquet(PARQUET)

## 3. Nettoyer / préparer le texte

Pour compter les mots de façon utile, on enlève le "bruit" :
- on met tout en **minuscules** (« Politique » et « politique » = le même mot)
- on enlève la **ponctuation** et les **chiffres**
- on enlève les **mots-outils** (« le », « de », « et »…) : ils sont partout et ne disent rien du sujet de l'article. On les appelle *stop words*.

In [ ]:
STOP_FR = set("""
au aux avec ce ces dans de des du elle en et eux il ils je la le les leur lui ma mais
me meme mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se ses son sur
ta te tes toi ton tu un une vos votre vous c d j l m n s t y est sont a ont etre avoir
fait faire plus tout tous toute toutes cette cet comme aussi alors donc car si entre
depuis apres avant deux trois ans an dont leurs ete tres bien plusieurs selon contre
vers chez sans sous quand encore aussi non oui ceux celle celles celui
""".split())

In [ ]:
import re

# Garde seulement les lettres (y compris accents et ligatures œ/æ), met en minuscules,
# enlève les mots-outils et les mots de moins de 3 lettres.
# NB : sans œ/æ, « cœur » ou « œuvre » seraient coupés en morceaux.
motif = re.compile(r"[a-zà-ÿœæ]+")

def nettoyer(texte):
    mots = motif.findall(str(texte).lower())
    mots = (m for m in mots if m not in STOP_FR and len(m) > 2)
    return " ".join(mots)

In [ ]:
# On applique le nettoyage à tout le corpus 
df["texte"] = df["contenu"].map(nettoyer)
df[["contenu", "texte"]].head(3)

## 4. TF-IDF

TF-IDF répond à : *« ce mot est-il fréquent dans CET article ET rare dans l'ensemble du corpus ? »*
Un mot qui passe ce double filtre est **caractéristique** de l'article — c'est un bon indicateur de sujet.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriseur = TfidfVectorizer(
    max_features=5000,   # on garde les 5000 mots les plus présents
    min_df=5,            # un mot doit apparaître dans au moins 5 articles
    max_df=0.5,          # ignore les mots présents dans plus de 50% des articles (trop banals)
)

matrice = vectoriseur.fit_transform(df["texte"])
vocab = vectoriseur.get_feature_names_out()

print("matrice :", matrice.shape)

## 5. Premier regard temporel : un mot dans le temps

Dans la matrice TF-IDF, chaque mot a sa colonne (un score par article). Pour suivre un mot,
on extrait sa colonne, on colle à chaque score l'année de l'article, puis on fait la
moyenne par année :

| année | score moyen |
|-------|-------------|
| 2022  | 0.002       |
| 2023  | 0.019       |
| 2024  | 0.028       |
| 2025  | 0.030       |



In [ ]:
def graphe_tf_idf(mot, ax=None, par="Y"):
    seul = ax is None
    if seul:
        fig, ax = plt.subplots(figsize=(8, 4))

    if mot not in vocab:
        ax.set_title(f"« {mot} » absent du vocabulaire")
        ax.axis("off")
        return

    col = list(vocab).index(mot)
    cle = df["date"].dt.tz_localize(None).dt.to_period(par).astype(str)
    serie = pd.DataFrame({
        "periode": cle.values,
        "score": matrice[:, col].toarray().ravel(),
    })
    par_periode = serie.groupby("periode")["score"].mean()
    par_periode.plot(ax=ax, title=f"« {mot} » (TF-IDF moyen / {par})")

    if seul:
        plt.tight_layout()
        plt.show()


In [ ]:
MOTS = ["oqtf", "immigration", "europe", "souveraineté"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for mot, ax in zip(MOTS, axes.ravel()):
    graphe_tf_idf(mot, ax)
fig.tight_layout()


## 6. 1-gramme

Précédemment on utilisait **TF-IDF**, qui pondère chaque mot selon sa rareté dans le corpus.
Ici on fait plus simple : on compte combien de fois le mot apparaît, sans pondération.

Pour suivre un mot dans le temps, on regarde ici son nombre moyen d'occurrences par article et par an.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt


compteur = CountVectorizer(
    max_features=5000,   
    min_df=5,
    max_df=0.5,
)
comptes = compteur.fit_transform(df["texte"])
vocab_compte = compteur.get_feature_names_out()
print("matrice :", comptes.shape)

In [ ]:
def graphe_1_gramme(mot, ax=None, par="Y"):
    seul = ax is None
    if seul:
        fig, ax = plt.subplots(figsize=(8, 4))

    if mot not in vocab_compte:
        ax.set_title(f"« {mot} » absent du vocabulaire")
        ax.axis("off")
        return

    col = list(vocab_compte).index(mot)
    cle = df["date"].dt.tz_localize(None).dt.to_period(par).astype(str)
    serie = pd.DataFrame({
        "periode": cle.values,
        "occurrences": comptes[:, col].toarray().ravel(),
    })
    par_periode = serie.groupby("periode")["occurrences"].sum()
    par_periode.plot(ax=ax, title=f"« {mot} » (occurrences totales / {par})")

    if seul:
        plt.tight_layout()
        plt.show()

In [ ]:
MOTS = ["oqtf", "immigration", "europe", "souveraineté"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for mot, ax in zip(MOTS, axes.ravel()):
    graphe_1_gramme(mot, ax)
fig.tight_layout()

## 7. 2-grammes

Un mot seul est parfois ambigu : « europe » peut désigner le continent,
l'équipe de foot, la radio Europe 1… La paire « union europeenne » est beaucoup plus précise sur le sujet.
Les bigrammes capturent donc des expressions et pas seulement des mots isolés.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

# Comptage brut, mais sur des paires de mots (bigrammes) : ngram_range=(2, 2).
compteur2 = CountVectorizer(
    min_df=5,
    max_df=0.5,
    ngram_range=(2, 2),   # uniquement des paires de mots
)
comptes2 = compteur2.fit_transform(df["texte"])
vocab2 = compteur2.get_feature_names_out()
print("matrice :", comptes2.shape, "(articles, bigrammes)")

In [ ]:
def graphe_2_grammes(expr, ax=None, par="Y"):
    seul = ax is None
    if seul:
        fig, ax = plt.subplots(figsize=(8, 4))

    if expr not in vocab2:
        ax.set_title(f"« {expr} » absent du vocabulaire")
        ax.axis("off")
        return

    col = list(vocab2).index(expr)
    cle = df["date"].dt.tz_localize(None).dt.to_period(par).astype(str)
    serie = pd.DataFrame({
        "periode": cle.values,
        "occurrences": comptes2[:, col].toarray().ravel(),
    })
    par_periode = serie.groupby("periode")["occurrences"].sum()
    par_periode.plot(ax=ax, title=f"« {expr} » (occurrences totales / {par})")

    if seul:
        plt.tight_layout()
        plt.show()


In [ ]:
EXPRESSIONS = ["loi immigration", "politique migratoire", "marine pen", "jordan bardella"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for expr, ax in zip(EXPRESSIONS, axes.ravel()):
    graphe_2_grammes(expr, ax, par="M")
fig.tight_layout()

## 8. Séries temporelles sur les bases ngram (Le Figaro, Les Échos)

In [ ]:
import sqlite3
import pandas as pd

BASES = {
    "Le Figaro": "../data/corpus/lefigaro_ngram.db",
    "Les Échos": "../data/corpus/lesechos_ngram.db",
}

In [ ]:
def serie_mot(mot, base_path):
    conn = sqlite3.connect(base_path)
    row = conn.execute("SELECT id FROM token WHERE word = ?", (mot,)).fetchone()
    if row is None:
        conn.close()
        return pd.DataFrame(columns=["mois", "n"])

    serie = pd.read_sql("""
        SELECT date / 100 AS mois, SUM(n) AS n
        FROM unigram
        WHERE w1 = ?
        GROUP BY mois
        ORDER BY mois
    """, conn, params=(row[0],))
    conn.close()
    return serie

In [ ]:
def graphe_ngram_db(mot, ax=None):
    seul = ax is None
    if seul:
        fig, ax = plt.subplots(figsize=(8, 4))

    for nom, chemin in BASES.items():
        serie = serie_mot(mot, chemin)
        if serie.empty:
            continue
        dates = pd.to_datetime(serie["mois"].astype(str), format="%Y%m")
        ax.plot(dates, serie["n"], label=nom)

    ax.set_title(f"« {mot} » (occurrences / mois)")
    ax.legend()

    if seul:
        plt.tight_layout()
        plt.show()

In [ ]:
MOTS_NGRAM = ["immigration", "europe", "souveraineté", "inflation"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for mot, ax in zip(MOTS_NGRAM, axes.ravel()):
    graphe_ngram_db(mot, ax)
fig.tight_layout()